<a href="https://colab.research.google.com/github/danielpazrosseboe/MasterDRC/blob/main/Export_GEE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ====================== EXACT-SCHEMA TFRECORD EXPORT (FIXED) ======================
# Exports post-2016 clusters from clusters_yeh_spec.csv to Google Drive folder
# 'dhs_tfrecords_raw' with Yeh-style schema:
# - Float list (len 65025) per band: BLUE,GREEN,RED,NIR,SWIR1,SWIR2,TEMP1,LON,LAT,NIGHTLIGHTS
# - Scalars: cluster_index (int), year (int), lon (float), wealthpooled (float), country (bytes)
# - No scalar 'lat'
# ================================================================================

import math, time
from typing import Dict, Tuple, Any

import ee
import pandas as pd
from tqdm.auto import tqdm

# ---- Authenticate/initialize if needed ----
ee.Authenticate()
ee.Initialize(project='master-thesis-measles')

CSV_PATH           = "/content/drive/My Drive/Master_Thesis/Surveys/clusters_yeh_spec.csv"
DRIVE_FOLDER       = "dhs_tfrecords_raw"
SCALE              = 30
EXPORT_TILE_RADIUS = 127   # 255x255
CHUNK_SIZE         = 10    # lower if memory errors
POLL_SECONDS       = 20

BANDS = ['BLUE','GREEN','RED','NIR','SWIR1','SWIR2','TEMP1','LON','LAT','NIGHTLIGHTS']

def three_year_window(y: int) -> Tuple[str, str]:
    if   2015 <= y <= 2017: return "2015-01-01", "2017-12-31"
    elif 2018 <= y <= 2020: return "2018-01-01", "2020-12-31"
    elif 2021 <= y <= 2023: return "2021-01-01", "2023-12-31"
    elif 2024 <= y <= 2026: return "2024-01-01", "2026-12-31"
    else:                   return f"{y-1}-01-01", f"{y+1}-12-31"

def composite_viirs(start: str, end: str) -> ee.Image:
    return (ee.ImageCollection('NOAA/VIIRS/DNB/MONTHLY_V1/VCMSLCFG')
            .filterDate(start, end)
            .median()
            .select(['avg_rad'], ['NIGHTLIGHTS']))

L8_SR = 'LANDSAT/LC08/C02/T1_L2'
L9_SR = 'LANDSAT/LC09/C02/T1_L2'

def mask_c2_qa(img: ee.Image) -> ee.Image:
    qa = img.select('QA_PIXEL')
    keep = (qa.bitwiseAnd(1 << 3).eq(0)
            .And(qa.bitwiseAnd(1 << 4).eq(0))
            .And(qa.bitwiseAnd(1 << 5).eq(0))
            .And(qa.bitwiseAnd(1 << 7).eq(0)))
    return img.updateMask(keep)

def landsat_c2_sr_stack(roi: ee.Geometry, start: str, end: str) -> ee.Image:
    sel = ['SR_B2','SR_B3','SR_B4','SR_B5','SR_B6','SR_B7','ST_B10']
    c8 = (ee.ImageCollection(L8_SR).filterBounds(roi).filterDate(start, end)
          .map(mask_c2_qa).select(sel))
    c9 = (ee.ImageCollection(L9_SR).filterBounds(roi).filterDate(start, end)
          .map(mask_c2_qa).select(sel))
    merged = c8.merge(c9)
    return (merged.median()
            .rename(['BLUE','GREEN','RED','NIR','SWIR1','SWIR2','TEMP1']))

def add_lonlat(img: ee.Image) -> ee.Image:
    latlon = ee.Image.pixelLonLat().select(['longitude','latitude'], ['LON','LAT'])
    return img.addBands(latlon)

def df_to_features(df: pd.DataFrame) -> ee.FeatureCollection:
    feats = []
    for _, r in df.iterrows():
        geom = ee.Geometry.Point([float(r['lon']), float(r['lat'])])
        feats.append(ee.Feature(geom, {
            'country':        str(r['country']),
            'year':           int(r['year']),
            'cluster_index':  int(r['cluster']),
            'lon':            float(r['lon']),
            'wealthpooled':   float(r['mean_pca_wealth'])
        }))
    return ee.FeatureCollection(feats)

def make_array_image(img: ee.Image, radius_px: int) -> ee.Image:
    """Create ONE image with per-band array bands, each renamed to the band name."""
    kern = ee.Kernel.square(radius=radius_px, units='pixels')
    arr_imgs = []
    for b in BANDS:
        arr_imgs.append(img.select([b]).neighborhoodToArray(kern).rename(b))
    return ee.Image.cat(arr_imgs)

def export_chunk(yr: int,
                 base_img: ee.Image,
                 points_fc: ee.FeatureCollection,
                 folder: str,
                 fname: str) -> ee.batch.Task:
    start, end = three_year_window(yr)
    img = add_lonlat(base_img).addBands(composite_viirs(start, end)).select(BANDS)
    arr_img = make_array_image(img, EXPORT_TILE_RADIUS)

    # Sample all band-arrays once; copy scalar props from the point
    def _sample_point(f):
        s = (arr_img.sample(
                region=f.geometry(),
                scale=SCALE,
                projection='EPSG:3857',
                numPixels=1,
                dropNulls=False,
                tileScale=12)
             .first())
        out = ee.Feature(None).copyProperties(f, ['country','year','cluster_index','lon','wealthpooled'])
        # s has array properties named exactly as BANDS; set them all at once
        return out.setMulti(s.toDictionary(ee.List(BANDS)))

    samples = points_fc.map(_sample_point)

    task = ee.batch.Export.table.toDrive(
        collection=samples,
        description=fname,
        folder=folder,
        fileNamePrefix=fname,
        fileFormat='TFRecord',
        selectors=['country','year','cluster_index','lon','wealthpooled'] + BANDS
    )
    task.start()
    return task

def run_exports():
    df = pd.read_csv(CSV_PATH)
    need = {'country','year','cluster','lat','lon','mean_pca_wealth'}
    missing = need - set(df.columns)
    if missing:
        raise ValueError(f"{CSV_PATH} missing columns: {sorted(missing)}")

    df['year'] = pd.to_numeric(df['year'], errors='coerce').astype('Int64')
    df = df.dropna(subset=['lat','lon','year','cluster'])
    df = df.loc[df['year'] >= 2016].copy()
    if df.empty:
        raise SystemExit("No rows with year >= 2016 in clusters_yeh_spec.csv")

    groups = list(df.groupby(['country','year']))
    tasks: Dict[Tuple[Any], ee.batch.Task] = {}

    for (cc, yr), g in tqdm(groups, desc="Exporting (country,year)"):
        g = g.reset_index(drop=True)
        roi = ee.Geometry.MultiPoint(g[['lon','lat']].values.tolist())
        start, end = three_year_window(int(yr))
        base = landsat_c2_sr_stack(roi, start, end)

        n = len(g)
        n_chunks = math.ceil(n / CHUNK_SIZE) if CHUNK_SIZE else 1
        for i in range(n_chunks):
            sl = slice(i*CHUNK_SIZE, min((i+1)*CHUNK_SIZE, n))
            fc = df_to_features(g.iloc[sl].copy())
            fname = f"{cc}_{int(yr)}_{i:02d}"
            t = export_chunk(int(yr), base, fc, DRIVE_FOLDER, fname)
            tasks[(cc, int(yr), i)] = t
            time.sleep(0.25)

    done = {
        ee.batch.Task.State.COMPLETED,
        ee.batch.Task.State.FAILED,
        ee.batch.Task.State.CANCEL_REQUESTED,
        ee.batch.Task.State.CANCELLED
    }
    remaining = list(tasks.keys())
    bar = tqdm(total=len(remaining), desc="Task status")
    while remaining:
        new_remaining = []
        for k in remaining:
            st = tasks[k].status()
            state = st.get('state', 'UNKNOWN')
            if state in done:
                bar.update(1)
                if state == ee.batch.Task.State.FAILED:
                    bar.write(f"{k} → FAILED: {st.get('error_message','')}")
                else:
                    bar.write(f"{k} → {state}")
            else:
                new_remaining.append(k)
        remaining = new_remaining
        if remaining:
            time.sleep(POLL_SECONDS)
    bar.close()
    print("All exports finished (or reached a terminal state).")

# Kick off
run_exports()
